In [1]:
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
from google.colab import files

uploaded = files.upload()

video_path = list(uploaded.keys())[0]

print("Video uploaded:", video_path)

Saving WhatsApp Video 2026-10-30 at 5.15.45 PM.mp4 to WhatsApp Video 2026-10-30 at 5.15.45 PM.mp4
Video uploaded: WhatsApp Video 2026-10-30 at 5.15.45 PM.mp4


In [3]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO("yolo11n-pose.pt")

print("YOLO Pose Model Loaded Successfully")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLO Pose Model Loaded Successfully


In [4]:
import cv2
import numpy as np

cap = cv2.VideoCapture(video_path)

# Video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Output video
output_path = "/content/dancing_pose_output.mp4"

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

# YOLO human skeleton connections (COCO format)
skeleton = [
    (5, 6),
    (5, 7), (7, 9),
    (6, 8), (8, 10),
    (5, 11), (6, 12),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16)
]

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    # Black background
    black_frame = np.zeros((height, width, 3), dtype=np.uint8)

    # YOLO pose prediction
    results = model(frame, conf=0.4, verbose=False)

    for result in results:

        if result.keypoints is None:
            continue

        keypoints = result.keypoints.xy.cpu().numpy()

        for person in keypoints:

            # Draw skeleton lines
            for start, end in skeleton:

                x1, y1 = person[start]
                x2, y2 = person[end]

                if x1 > 0 and y1 > 0 and x2 > 0 and y2 > 0:

                    cv2.line(
                        black_frame,
                        (int(x1), int(y1)),
                        (int(x2), int(y2)),
                        (255, 255, 255),
                        3
                    )

            # Draw joints
            for x, y in person:

                if x > 0 and y > 0:

                    cv2.circle(
                        black_frame,
                        (int(x), int(y)),
                        4,
                        (255, 255, 255),
                        -1
                    )

    out.write(black_frame)

cap.release()
out.release()

print("Pose estimation completed!")

Pose estimation completed!


In [7]:
!ls -lh /content/dancing_pose_output.mp4

-rw-r--r-- 1 root root 6.9M Oct 30 11:57 /content/dancing_pose_output.mp4


In [16]:
from IPython.display import Video, display

display(Video(
    "/content/final_output.mp4",
    embed=True,
    width=800
))